# Deployment

Now, we'll cover how to deploy our LangGraph agent as a **Databricks Model Serving endpoint** — packaging the graph with MLflow and serving it via the same OpenAI-compatible API we've been using throughout the course.

## Concepts

The deployment approach uses:

**MLflow** —
- Tracks experiments and packages models as reusable artifacts
- The `mlflow.langchain` flavour natively supports LangGraph graphs
- Logs the graph as an MLflow model with all dependencies

**Databricks Model Serving** —
- Hosts MLflow models behind a REST endpoint
- Provides an OpenAI-compatible chat API (`/serving-endpoints/<name>/invocations`)
- Scales automatically, supports GPU, and integrates with workspace auth

**Workflow**:
1. Define the LangGraph agent (same as notebook 5/6)
2. Log it to MLflow with `mlflow.langchain.log_model()`
3. Register the model in Unity Catalog (or the workspace registry)
4. Create a Model Serving endpoint pointing at that registered model
5. Invoke the endpoint with `openai.OpenAI` or `ChatOpenAI` — just like the rest of our course

## Step 1 — Define the Agent

We'll reuse the simple multiply/add agent from notebook 5. This cell defines the graph inline so the notebook is self-contained.

In [ ]:
%run ../langchain_common.py

In [ ]:
from langgraph.graph import StateGraph, MessagesState, START, END
from langchain_core.messages import HumanMessage, SystemMessage

def multiply(a: int, b: int) -> int:
    """Multiply a and b.

    Args:
        a: first int
        b: second int
    """
    return a * b

def add(a: int, b: int) -> int:
    """Adds a and b.

    Args:
        a: first int
        b: second int
    """
    return a + b

tools = [multiply, add]
llm_with_tools = llm.bind_tools(tools)

In [ ]:
from langgraph.prebuilt import ToolNode

def assistant(state: MessagesState):
    return {"messages": [llm_with_tools.invoke(state["messages"])]}

builder = StateGraph(MessagesState)
builder.add_node("assistant", assistant)
builder.add_node("tools", ToolNode(tools))
builder.add_edge(START, "assistant")

def route_tools(state: MessagesState):
    last = state["messages"][-1]
    if hasattr(last, "tool_calls") and last.tool_calls:
        return "tools"
    return END

builder.add_conditional_edges("assistant", route_tools, ["tools", END])
builder.add_edge("tools", "assistant")

graph = builder.compile()
print("Graph compiled successfully")

{'assistant_id': '228f9934-0cdd-5383-92c8-ee8422522cc2',
 'graph_id': 'router',
 'config': {},
 'context': {},
 'metadata': {'created_by': 'system'},
 'name': 'router',
 'created_at': '2026-04-16T15:43:58.055650+00:00',
 'updated_at': '2026-04-16T15:43:58.055650+00:00',
 'version': 1,
 'description': None}

In [ ]:
# Quick sanity check — invoke locally before packaging
result = graph.invoke({"messages": [HumanMessage(content="Multiply 3 by 2.")]})
for msg in result["messages"]:
    msg.pretty_print()

## Step 2 — Log the Agent to MLflow

We use `mlflow.langchain.log_model()` to package the compiled graph. MLflow will:
- Capture the graph definition and all its dependencies
- Store it as a versioned artifact we can later register and serve

In [ ]:
import mlflow

mlflow.set_experiment("wk5-deployment")

with mlflow.start_run(run_name="langgraph-agent") as run:
    model_info = mlflow.langchain.log_model(
        lc_model=graph,
        artifact_path="langgraph_agent",
        input_example={"messages": [{"role": "user", "content": "Add 2 and 3."}]},
    )
    print(f"Model logged: {model_info.model_uri}")
    print(f"Run ID: {run.info.run_id}")

{'content': 'Multiply 3 by 2.', 'additional_kwargs': {}, 'response_metadata': {}, 'type': 'human', 'name': None, 'id': '9c5db9dc-7f45-490b-93de-c6c4e044fe80'}
{'content': '', 'additional_kwargs': {'refusal': None}, 'response_metadata': {'token_usage': {'completion_tokens': 18, 'prompt_tokens': 59, 'total_tokens': 77, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4.1-mini-2025-04-14', 'system_fingerprint': 'fp_b6f445fc1c', 'id': 'chatcmpl-DVJDjULwae7DpCphlaqeVtR6S2vwT', 'service_tier': 'default', 'prompt_filter_results': [{'prompt_index': 0, 'content_filter_results': {'hate': {'filtered': False, 'severity': 'safe'}, 'jailbreak': {'filtered': False, 'detected': False}, 'self_harm': {'filtered': False, 'severity': 'safe'}, 'sexual': {'filtered': False, 'severity': 'safe'}, 'violence': 

## Step 3 — Load and Test the Logged Model Locally

Before deploying, we reload the model from MLflow and verify it produces correct output.

In [ ]:
loaded_model = mlflow.langchain.load_model(model_info.model_uri)
result = loaded_model.invoke({"messages": [HumanMessage(content="What is 5 + 7?")]})
for msg in result["messages"]:
    msg.pretty_print()

## Step 4 — Register the Model

Register the logged model in the MLflow Model Registry. On Databricks this would target Unity Catalog (e.g. `catalog.schema.model_name`). Locally we use the workspace registry.

In [ ]:
from mlflow import MlflowClient

client = MlflowClient()
model_name = "langgraph_agent"

# Register the model (creates the registered model if it doesn't exist)
mv = client.create_model_version(
    name=model_name,
    source=model_info.model_uri,
    run_id=run.info.run_id,
)
print(f"Registered model '{model_name}' version {mv.version}")

## Step 5 — Deploy to Databricks Model Serving

On Databricks, you would create a **Model Serving endpoint** pointing at the registered model. The cell below shows the API call to create the endpoint.

> **Note**: This requires running in a Databricks workspace with appropriate permissions. The endpoint will be available at `{DATABRICKS_HOST}/serving-endpoints/langgraph-agent/invocations`.

```python
# Run this in a Databricks notebook or with workspace-level permissions:
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()

w.serving_endpoints.create(
    name="langgraph-agent",
    config={
        "served_entities": [{
            "entity_name": "catalog.schema.langgraph_agent",  # Unity Catalog path
            "entity_version": "1",
            "workload_size": "Small",
            "scale_to_zero_enabled": True,
        }]
    }
)
```

Once the endpoint is ready (status = `READY`), you can call it just like any other Databricks model serving endpoint.

## Step 6 — Call the Deployed Endpoint

Once the serving endpoint is live, we interact with it using the same OpenAI-compatible client pattern we've used throughout the course.

In [ ]:
import openai

# Point at the deployed endpoint (same pattern as all our other notebooks)
serving_client = openai.OpenAI(
    api_key=DATABRICKS_TOKEN,
    base_url=f"{DATABRICKS_HOST}/serving-endpoints",
)

response = serving_client.chat.completions.create(
    model="langgraph-agent",  # name of your serving endpoint
    messages=[{"role": "user", "content": "Multiply 3 by 2."}],
)
print(response.choices[0].message.content)

## Summary

| Step | Action |
|------|--------|
| 1 | Define LangGraph agent (graph with tools) |
| 2 | Log to MLflow with `mlflow.langchain.log_model()` |
| 3 | Verify locally by reloading the model |
| 4 | Register in Model Registry / Unity Catalog |
| 5 | Create a Databricks Model Serving endpoint |
| 6 | Call the endpoint via OpenAI-compatible API |

This gives us a production-ready deployment that:
- Scales automatically with traffic
- Integrates with Databricks workspace authentication
- Supports the same `openai.OpenAI` client interface used throughout the course
- Provides built-in monitoring via MLflow and Databricks serving metrics